# Chapitre 12 — Synthèse : préparer un jeu de données pour un modèle

## Le problème

La boutique veut anticiper quels clients vont commander au dernier trimestre 2025, pour cibler une campagne. Un modèle de classification pourrait l'aider, à condition de lui fournir une table où :

- **chaque ligne est un client** ;
- **chaque colonne est une variable numérique** connue *avant* la période à prédire ;
- une colonne **cible** indique ce qui s'est passé ensuite (a commandé ou non) ;
- les données sont séparées en **entraînement** et **test** sans que l'un n'influence l'autre.

Ce chapitre ne construit pas le modèle : il produit la matrice qui l'alimente. Il mobilise presque toutes les notions du cours.

In [ ]:
import numpy as np
import pandas as pd

clients = pd.read_csv("../donnees/demo/clients.csv", parse_dates=["date_inscription"])
produits = pd.read_csv("../donnees/demo/produits.csv")
commandes = pd.read_csv("../donnees/demo/commandes.csv", parse_dates=["date_commande"])

## 1. Fixer la frontière temporelle

Les variables doivent être calculées sur le passé, la cible sur le futur. Sinon, le modèle « verrait » la réponse dans ses variables : c'est une **fuite de données** (*data leakage*), qui donne d'excellents résultats en test et de très mauvais en production.

```
  2025-01-01                         2025-10-01                 2025-12-31
      │───────── variables (passé) ────────│──── cible (futur) ────│
```

In [ ]:
coupure = pd.Timestamp("2025-10-01")
passe = commandes[commandes["date_commande"] < coupure]
futur = commandes[commandes["date_commande"] >= coupure]
len(passe), len(futur)

## 2. Nettoyer et enrichir les commandes passées

On ne garde que les commandes de clients connus et non annulées, et on calcule le montant.

In [ ]:
passe = (passe
         .merge(produits[["produit_id", "categorie", "prix"]], on="produit_id", how="left", validate="many_to_one")
         .assign(montant=lambda d: d["quantite"] * d["prix"])
         .query("statut != 'annulée'"))
passe = passe[passe["client_id"].isin(clients["client_id"])]
passe.shape

## 3. Construire les variables par client

Une agrégation par client, des variables temporelles, et un tableau croisé pour la répartition par canal.

In [ ]:
agregats = passe.groupby("client_id").agg(
    nb_commandes=("commande_id", "count"),
    ca_total=("montant", "sum"),
    panier_moyen=("montant", "mean"),
    nb_categories=("categorie", "nunique"),
    derniere_commande=("date_commande", "max"),
)
agregats["jours_depuis_derniere"] = (coupure - agregats["derniere_commande"]).dt.days
agregats = agregats.drop(columns="derniere_commande")

parts_canal = pd.crosstab(passe["client_id"], passe["canal"], normalize="index").add_prefix("part_")
parts_canal.head(3)

## 4. Assembler la table des clients

On part de **tous** les clients (jointure `left`) : un client sans commande passée est une information, pas une ligne à perdre.

In [ ]:
table = (clients
         .merge(agregats, left_on="client_id", right_index=True, how="left", validate="one_to_one")
         .merge(parts_canal, left_on="client_id", right_index=True, how="left", validate="one_to_one"))
table["anciennete_jours"] = (coupure - table["date_inscription"]).dt.days
table.head()

> **Mini-problème.** Certaines colonnes contiennent maintenant des `NaN`. Lesquels doivent devenir 0, et lesquels demandent un autre traitement ? Regardez `table.isna().sum()` et raisonnez colonne par colonne.

In [ ]:
table.isna().sum()

- `nb_commandes`, `ca_total`, `nb_categories`, les parts de canal : un client sans commande a **zéro** commande. Le `NaN` vient de la jointure, sa vraie valeur est 0.
- `panier_moyen` : pas de commande, pas de panier. On met 0 et on ajoute un indicateur `a_deja_commande`.
- `jours_depuis_derniere` : la valeur n'existe pas. On peut la remplacer par une valeur élevée (l'ancienneté) pour signifier « jamais ».
- `age` : une vraie donnée inconnue, que l'on imputera **après** la séparation entraînement / test.

In [ ]:
zeros = ["nb_commandes", "ca_total", "panier_moyen", "nb_categories"] + list(parts_canal.columns)
table[zeros] = table[zeros].fillna(0)
table["a_deja_commande"] = (table["nb_commandes"] > 0).astype(int)
table["jours_depuis_derniere"] = table["jours_depuis_derniere"].fillna(table["anciennete_jours"])

## 5. La cible

In [ ]:
table["cible"] = table["client_id"].isin(futur["client_id"]).astype(int)
table["cible"].value_counts()

## 6. Encoder les variables catégorielles

In [ ]:
table = pd.get_dummies(table, columns=["segment", "ville"], drop_first=True, dtype=int)
table.columns.tolist()

## 7. Séparer entraînement et test, puis imputer

La séparation se fait **avant** toute statistique calculée sur les données (imputation, normalisation). Ici, une séparation aléatoire reproductible : 70 % des clients pour l'entraînement.

In [ ]:
rng = np.random.default_rng(42)
melange = rng.permutation(table.index)
n_train = int(len(table) * 0.7)
train = table.loc[melange[:n_train]].copy()
test = table.loc[melange[n_train:]].copy()

mediane_age = train["age"].median()          # statistique calculée sur l'entraînement uniquement
train["age"] = train["age"].fillna(mediane_age)
test["age"] = test["age"].fillna(mediane_age)
len(train), len(test), mediane_age

> **Pourquoi ?** Que se passerait-il si `mediane_age` était calculée sur `table` entière, avant la séparation ?

## 8. La matrice finale

On retire les colonnes qui ne sont pas des variables : identifiant, prénom, date brute, cible.

In [ ]:
a_exclure = ["client_id", "prenom", "date_inscription", "cible"]
variables = [c for c in table.columns if c not in a_exclure]

X_train = train[variables].to_numpy(dtype=float)
y_train = train["cible"].to_numpy()
X_test = test[variables].to_numpy(dtype=float)
y_test = test["cible"].to_numpy()

X_train.shape, X_test.shape, np.isnan(X_train).sum(), np.isnan(X_test).sum()

In [ ]:
pd.DataFrame(X_train, columns=variables).describe().T.round(2)

`X_train` et `y_train` sont exactement ce qu'attend un modèle de scikit-learn (`model.fit(X_train, y_train)`), ou un `Dataset` PyTorch après conversion en tenseurs.

## La chaîne complète

| Étape | Chapitre | Outils |
|---|---|---|
| Charger | 3 | `read_csv`, `parse_dates` |
| Filtrer sur le temps | 4, 5 | masques, `Timestamp` |
| Joindre et contrôler | 8 | `merge`, `validate`, `isin` |
| Agréger par entité | 6, 7 | `groupby().agg()`, `crosstab` |
| Traiter les manquants avec du sens | 5 | `fillna` colonne par colonne |
| Encoder | 9 | `get_dummies` |
| Séparer avant d'imputer | 5, 12 | `permutation`, `median` sur train |
| Produire la matrice | 2 | `to_numpy(dtype=float)` |

## Exercices

- Exercice supplémentaire **S12** : construire une table de variables à partir des données JO.

## À retenir

- Une ligne par entité à prédire, une colonne numérique par variable, une cible.
- Les variables viennent du passé, la cible du futur : une frontière temporelle explicite.
- Un `NaN` issu d'une jointure n'a pas le même sens qu'une donnée inconnue.
- Séparer entraînement et test **avant** de calculer une statistique d'imputation ou de normalisation.
- `to_numpy(dtype=float)` produit la matrice finale ; vérifier qu'elle ne contient aucun `NaN`.